# Initialization

In [0]:
from pyspark.sql import functions as F

In [0]:
SALES_RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "shipping_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

# Reading From bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Transformations

## Renaming the Columns

In [0]:
for old_name, new_name in SALES_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Date Conversion

In [0]:
df = df.withColumn(
    "order_date",
    F.when(
        F.col("order_date").cast("string") == "0",
        None
    ).otherwise(F.col("order_date"))
)

df = df.withColumn(
    "order_date",
    F.when(
        F.length(F.col("order_date").cast("string")) == 8,
        F.to_date(
            F.col("order_date").cast("string"),
            "yyyyMMdd"
        )
    ).otherwise(None)
)

In [0]:
df = df.withColumn(
    "shipping_date",
    F.when(
        F.col("shipping_date").cast("string") == "0",
        None
    ).otherwise(F.col("shipping_date"))
)

df = df.withColumn(
    "shipping_date",
    F.when(
        F.length(F.col("shipping_date").cast("string")) == 8,
        F.to_date(
            F.col("shipping_date").cast("string"),
            "yyyyMMdd"
        )
    ).otherwise(None)
)

In [0]:
df = df.withColumn(
    "due_date",
    F.when(
        F.col("due_date").cast("string") == "0",
        None
    ).otherwise(F.col("due_date"))
)

df = df.withColumn(
    "due_date",
    F.when(
        F.length(F.col("due_date").cast("string")) == 8,
        F.to_date(
            F.col("due_date").cast("string"),
            "yyyyMMdd"
        )
    ).otherwise(None)
)

## Correcting Sales Amount

In [0]:
df = df.withColumn(
    "sales_amount",
    F.when(
        (F.col("sales_amount").isNull()) |
        (F.col("sales_amount") <= 0) |
        (
            F.col("sales_amount") !=
            F.col("quantity") *
            F.abs(F.col("price"))
        ),
        F.col("quantity") *
        F.abs(F.col("price"))
    )
    .otherwise(F.col("sales_amount"))
)

## Correcting the Price

In [0]:
df = df.withColumn(
    "price",
    F.when(
        (F.col("price").isNull()) |
        (F.col("price") <= 0),
        F.col("sales_amount") /
        F.when(
            F.col("quantity") == 0,
            None
        ).otherwise(F.col("quantity"))
    )
    .otherwise(F.col("price"))
)

## Handling Correct Order Dates

In [0]:
df = df.withColumn(
    "order_date",
    F.when(
        (F.col("order_date") > F.col("shipping_date")) |
        (F.col("order_date") > F.col("due_date")),
        None
    )
    .otherwise(F.col("order_date"))
)

## Selecting Columns

In [0]:
df = df.select(
    "order_number",
    "product_key",
    "customer_id",
    "order_date",
    "shipping_date",
    "due_date",
    "sales_amount",
    "quantity",
    "price"
)

# Writing into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_sales")
)